In [1]:
!pip install transformers torch datasets -q

In [2]:
import torch
from transformers import pipeline

torch.manual_seed(42)
print('Torch version:', torch.__version__)

In [3]:
# Toy embeddings for 3 tokens
X = torch.tensor([[1., 0., 1., 0.],
                  [0., 2., 0., 2.],
                  [1., 1., 0., 0.]])  # shape (seq, dim)

# Random weight matrices (fixed seed above)
W_q = torch.randn(4, 4)
W_k = torch.randn(4, 4)
W_v = torch.randn(4, 4)

Q = X @ W_q
K = X @ W_k
V = X @ W_v

dk = K.size(-1)
scores = (Q @ K.T) / dk**0.5
weights = scores.softmax(dim=-1)
attended = weights @ V

print('Q shape:', Q.shape)
print('Attention weights (row = query token):')
print(weights)
print('Output embeddings after attention:')
print(attended)

## Pipeline: Sentiment Analysis

In [4]:
sentiment = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')
texts = [
    'Transformers make NLP much easier.',
    'I am not sure this movie was worth my time.',
    'The food was decent but the service was slow.'
]
for t in texts:
    print(t, '->', sentiment(t)[0])

## Pipeline: Masked Language Modeling (RoBERTa)

In [5]:
fill_mask = pipeline('fill-mask', model='distilroberta-base')
masked_sentence = 'Transformers are <mask> for many NLP tasks.'
for pred in fill_mask(masked_sentence)[:5]:
    print(f"{pred['sequence']} (score={pred['score']:.4f})")

## Pipeline: Text Generation (DistilGPT2)

In [6]:
generator = pipeline('text-generation', model='distilgpt2')
prompt = 'In 2025, natural language models'
outputs = generator(prompt, max_length=40, num_return_sequences=2, do_sample=True, top_p=0.95, top_k=50)
for i, out in enumerate(outputs, 1):
    print(f"\nSample {i}: {out['generated_text']}")